# ROI tiles

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import contextily as cx
import matplotlib.pyplot as plt

In [ ]:
import evaspa
import evaspa.trishna as trishna
import evaspa.tiling as tiling

In [ ]:
outdir = "../report"
os.makedirs(outdir,exist_ok=True)

### Read TRISHNA grid

In [ ]:
trishna_grid = trishna.get_trishna_tiles()

In [ ]:
trishna_grid.head()

In [ ]:
len(trishna_grid)

### Read land mask

In [ ]:
land = trishna.get_land_mask()

In [ ]:
land.head()

In [ ]:
land_trishna_tiles = tiling.intersection(trishna_grid, land)

In [ ]:
len(land_trishna_tiles)

### Read orbits 

In [ ]:
orbits = trishna.get_trishna_orbits()

In [ ]:
orbits.head()

### Read ROI

In [ ]:
roi = gpd.read_file("roi_italy.shp")
roi = roi.set_crs(4326)

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.gca()
ax = roi.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                   )
cx.add_basemap(ax,crs=roi.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("ROI")
fig.savefig(os.path.join(outdir,"roi.png"))

### Select tiles

In [ ]:
roi_tiles = tiling.intersection(trishna_grid[["id","geometry"]],roi)

In [ ]:
# Filter
roi_tiles = roi_tiles[roi_tiles.overlap_percentage > 10]

In [ ]:
roi_tiles = gpd.GeoDataFrame(roi_tiles[['id']].merge(land_trishna_tiles, how='inner', on="id"))

In [ ]:
roi_tiles["land"] = roi_tiles["overlap_percentage"]/100

In [ ]:
roi_tiles.head()

In [ ]:
len(roi_tiles)

### Plot

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = plt.gca()
ax = roi_tiles.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="land", 
                    cmap="viridis", 
                    legend=True,
                    legend_kwds={
                        "shrink":.6
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("Percentage of land per tile")
fig.savefig(os.path.join(outdir,"roi_tiles_landpercentage.png"))

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = plt.gca()
ax = roi_tiles.plot(ax=ax, 
                    alpha=0.8, 
                    edgecolor="k",
                    column="utm_zone", 
                    categorical=True,
                    cmap="Pastel1", 
                    legend=True,
                    legend_kwds={
                        "loc":"lower right",
                    },
                   )
cx.add_basemap(ax,crs=roi_tiles.crs)
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_title("UTM zones")

In [ ]:
roi_tiles.columns

In [ ]:
roi_tiles[["id","utm_zone","epsg","geometry","land"]].to_file("roi_italy.gpkg",driver="GPKG")

In [ ]:
def plot_tiles(tile_ids
    ) -> None:
    fig = plt.figure(figsize=(6, 4))
    ax = plt.gca()
    gdf = roi_tiles[roi_tiles["id"].isin(tile_ids)]
    ax = gdf.plot(ax=ax, 
                        alpha=0.8, 
                        edgecolor="k",
                       )
    bb = gdf.bounds
    ax.set(xlim=(bb.minx.values[0] - 3, bb.maxx.values[0] + 3), ylim=(bb.miny.values[0] - 2, bb.maxy.values[0] + 2))
    cx.add_basemap(ax,crs=roi_tiles.crs)
    ax.set_ylabel("Latitude")
    ax.set_xlabel("Longitude")
    fig.savefig(os.path.join(outdir,f'plot_map_{"-".join(tile_ids)}.png'))

In [ ]:
plot_tiles(["32TLP"])